# Calculating global pasture inventories using above ground biomass

We want to use an above ground biomass dataset. This gives us the total amount of plant biomass at a given time available for consumption by ruminants. To convert this into available residues, the method will differ depending on what plant species we're looking at:

- For crops: We instead use the SPAM 2010 dataset - see notebooks 1.0 and 2.0
- For everything else: Remove cropped areas, urban areas and woody areas. For the remainder, assume all above ground land biomass is available as residue.

## Down-sample biomass data

The above-ground biomass carbon raster file is ~2.7GB, making it somewhat unwieldy to work with locally. We don't need it to be this big, as we are otherwise restricted by the size of our MODIS land class data. We'll start by re-sampling the biomass raster to be the same shape as our crop raster.

In [2]:
import rasterio

MODIS_LANDCOVER_RASTER = '../../data/processed/modis-land-cover-2006-4326.tif'

landcover_dataset = rasterio.open(MODIS_LANDCOVER_RASTER)
landcover_array = landcover_dataset.read(1)

In [3]:
landcover_array.shape

(3600, 7200)

In [4]:
CARBON_BIOMASS_RASTER = "../../data/raw/Global_Maps_C_Density_2010_1763/data/aboveground_biomass_carbon_2010.tif"

carbon_biomass_dataset = rasterio.open(CARBON_BIOMASS_RASTER)
carbon_biomass_array = carbon_biomass_dataset.read(1)

In [5]:
carbon_biomass_array.max()

3481

In [6]:
# Check for nodata values (should be none)
carbon_nodata = carbon_biomass_array == 65536
carbon_nodata.sum()

0

In [7]:
carbon_biomass_dataset.res

(0.00277777777777778, 0.0027777777777777805)

In [8]:
# Find a good down-sampling factor

carbon_biomass_array.shape[1] / landcover_array.shape[1]

18.0

In [9]:
# Define a re-sampling method

import rasterio
from rasterio.enums import Resampling

def resample_raster_file(src_file, resampling_method = Resampling.bilinear, upscale_factor=0.1):

    with rasterio.open(src_file) as dataset:
        
        print("Reading and resampling...")
        # resample data to target shape
        data = dataset.read(
            out_shape=(
                dataset.count,
                int(dataset.height * upscale_factor),
                int(dataset.width * upscale_factor)
            ),
            resampling=resampling_method
        )
        
        print("Scaling the transform...")
        # scale image transform
        transform = dataset.transform * dataset.transform.scale(
            (dataset.width / data.shape[-1]),
            (dataset.height / data.shape[-2])
        )
        
        print("Done")
        
        return data, transform

In [10]:
import numpy as np

RESAMPLED_CARBON_BIOMASS_RASTER = "../../data/processed/biomass_resampled.tif"
DOWNSCALE_FACTOR = 18

# Re-sample the biomass raster to crop raster (down-scale by a factor of 18)
biomass_arr, biomass_transform = resample_raster_file(CARBON_BIOMASS_RASTER, resampling_method=Resampling.average, upscale_factor=1/DOWNSCALE_FACTOR)

# Save processed data
biomass_dataset = rasterio.open(
    RESAMPLED_CARBON_BIOMASS_RASTER,
    "w",
    driver="GTiff",
    height=carbon_biomass_array.shape[0] / DOWNSCALE_FACTOR,
    width=carbon_biomass_array.shape[1] / DOWNSCALE_FACTOR,
    count=1,
    dtype=biomass_arr.dtype,
    crs='+proj=latlong',
    transform=biomass_transform
)
biomass_arr = np.squeeze(biomass_arr)
biomass_dataset.write(biomass_arr, 1)
biomass_dataset.close()

Reading and resampling...
Scaling the transform...
Done


In [18]:
# Checking that these results make sense (quantities are preserved)

In [16]:
(carbon_biomass_array.shape[0] * carbon_biomass_array.shape[1]) / (biomass_arr.shape[0] * biomass_arr.shape[1]) 

324.0062068965517

In [11]:
biomass_arr.sum()

1167774545

In [17]:
carbon_biomass_array.sum() / 324.0062068965517

1151660802.9862137

# Get carbon biomass data for grasslands only

Units are in Mg C/ha (tonnes C / ha)

Amount represents annual peak of biomass

Extents of grassland are the 'grassland' class in the (MODIS) Land Cover product for 2006

From MODIS user guide: "• Some grassland areas are classified as savannas (sparse forest)." https://lpdaac.usgs.gov/documents/101/MCD12_User_Guide_V6.pdf

What conversion figure to use for carbon from grasses?

https://bg.copernicus.org/articles/15/693/2018/bg-15-693-2018.pdf

0.4473

Note that this is for "oven dry" biomass https://www.fao.org/forestry/17111/en/

In [10]:
# In order to find the pixels of the carbon biomass raster which correspond to grassland pixels
# in the landcover raster, we first need to modify the carbon biomass raster so it has the 
# same bounds as the landcover data

print(landcover_dataset.bounds)
print(landcover_dataset.res)
print(landcover_dataset.shape)
print(biomass_dataset.bounds)

BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
(0.05, 0.05)
(3600, 7200)
BoundingBox(left=-180.00000000000006, bottom=-61.00277777777792, right=180.00000000000028, top=84.00000000000001)


In [11]:
!rio warp "../../data/processed/biomass_resampled.tif" "../../data/processed/biomass_resampled_warped_bounds.tif" --overwrite --dst-crs EPSG:4326 --bounds -180 -90 180 90 --res 0.05

In [12]:
carbon_biomass_dataset = rasterio.open("../../data/processed/biomass_resampled_warped_bounds.tif")
carbon_biomass_array = carbon_biomass_dataset.read(1)

In [13]:
print(carbon_biomass_dataset.bounds)
print(carbon_biomass_dataset.res)
print(carbon_biomass_array.shape)

BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
(0.05, 0.05)
(3600, 7200)


In [14]:
# Create a boolean matrix of grass values
grassland = landcover_array == 10.0

# Use only carbon biomass which corresponds to True values in the matrix
carbon_biomass_from_grassland_arr = grassland * carbon_biomass_array

# Convert to biomass from carbon biomass
ABOVE_GROUND_CARBON_MASS_FRACTION_FOR_GRASSES = 0.4473
biomass_from_grassland_arr = carbon_biomass_from_grassland_arr / ABOVE_GROUND_CARBON_MASS_FRACTION_FOR_GRASSES

In [15]:
# Save processed data

GRASSLAND_BIOMASS_RASTER = "../../data/processed/biomass_mg_per_ha_from_grassland.tif"

biomass_from_grassland_dataset = rasterio.open(
    GRASSLAND_BIOMASS_RASTER,
    "w",
    driver="GTiff",
    height=biomass_from_grassland_arr.shape[0],
    width=biomass_from_grassland_arr.shape[1],
    count=1,
    dtype=biomass_from_grassland_arr.dtype,
    crs='+proj=latlong',
    transform=carbon_biomass_dataset.transform
)

biomass_from_grassland_dataset.write(biomass_from_grassland_arr, 1)
biomass_from_grassland_dataset.close()

# Summarise

The `biomass_grom_grassland_arr` array is un-projected, i.e. the pixels are lat/lon based. This means the area (in meters) of each pixel will vary based on distance to the equator. The values in each cell are tonnes/hectare, so to get a tonnes production figure, we need to multiply the value of each cell by its area.

In [16]:
import numpy as np
import numpy.matlib
import rasterio

with rasterio.open(GRASSLAND_BIOMASS_RASTER) as dataset:
    rast = dataset.read(1)
    transform = dataset.transform
    pix_width = transform[0]
    
    ulX = transform[2]
    ulY = transform[5]
    
    rows = dataset.height
    cols = dataset.width
    
    lrX = ulX + transform[0] * cols
    lrY = ulY + transform[4] * rows

lats = np.linspace(ulY, lrY, rows + 1)

a = 6378137
b = 6356752.3142

# Degrees to radians
lats = lats * np.pi/180

# Intermediate vars
e = np.sqrt(1-(b/a)**2)
sinlats = np.sin(lats)
zm = 1 - e * sinlats
zp = 1 + e * sinlats

# Distance between meridians
q = pix_width/360

# Compute areas for each latitude in square km
areas_to_equator = np.pi * b**2 * ((2*np.arctanh(e*sinlats) / (2*e) + sinlats / (zp*zm))) / 10**6
areas_between_lats = np.diff(areas_to_equator)
areas_cells = np.abs(areas_between_lats) * q

areagrid = np.transpose(np.matlib.repmat(areas_cells,cols,1))

In [17]:
print(biomass_from_grassland_arr.shape)
print(areagrid.shape)
print(areagrid.max())
print(areagrid.min())

(3600, 7200)
(3600, 7200)
30.772676396604844
0.013608707657290829


In [18]:
# Convert area grid from square km to hectares (x 100) and multiple by our "biomass tonnes per hectare" array
tonnes_of_biomass_from_grass_arr = biomass_from_grassland_arr * areagrid * 100

In [21]:
print(tonnes_of_biomass_from_grass_arr.sum())

492280897474.3974


In [25]:
# Save processed data

GRASSLAND_BIOMASS_TONNES_RASTER = "../../data/processed/production_pit/biomass_from_grassland_tonnes.tif"

grassland_biomass_tonnes_dataset = rasterio.open(
    GRASSLAND_BIOMASS_TONNES_RASTER,
    "w",
    driver="GTiff",
    height=tonnes_of_biomass_from_grass_arr.shape[0],
    width=tonnes_of_biomass_from_grass_arr.shape[1],
    count=1,
    dtype=tonnes_of_biomass_from_grass_arr.dtype,
    crs='+proj=latlong',
    transform=carbon_biomass_dataset.transform
)

grassland_biomass_tonnes_dataset.write(tonnes_of_biomass_from_grass_arr, 1)
grassland_biomass_tonnes_dataset.close()